In [ ]:
import pandas as pd

# Upload the file first
from google.colab import files
uploaded = files.upload()

# Read it
import io
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]), encoding='latin1')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))
print("\nData types:")
print(df.dtypes)
print("\nNull counts:")
print(df.isnull().sum())

In [ ]:
!pip install xgboost lightgbm scikit-learn pandas mlflow

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('indian_roads_dataset.csv', encoding='latin1')
print("Original shape:", df.shape)

weather_rain_map = {
    'rain':120.0,'rainy':100.0,'heavy rain':160.0,
    'flood':180.0,'storm':140.0,'thunderstorm':150.0,
    'foggy':5.0,'fog':5.0,'mist':3.0,'haze':2.0,
    'cloudy':10.0,'overcast':8.0,'clear':0.0,
    'sunny':0.0,'windy':0.0,'snow':0.0,'sleet':20.0,
}

traffic_map  = {'low':0.2,'medium':0.5,'high':0.8}
visibility_map = {'low':85.0,'medium':65.0,'high':45.0}
cause_map = {
    'weather':1,'road condition':2,
    'drunk driving':1,'overloading':1,
    'human error':0,'mechanical':0,'speeding':0,
}

severity_map = {'fatal':1,'major':1,'minor':0,'none':0}

def map_rainfall(w):
    if pd.isna(w): return 0.0
    w = str(w).lower().strip()
    for key,val in weather_rain_map.items():
        if key in w: return val
    return 0.0

def map_cause(c):
    if pd.isna(c): return 0
    c = str(c).lower().strip()
    for key,val in cause_map.items():
        if key in c: return val
    return 0

city_highway_map = {
    'delhi':'NH-44','nagpur':'NH-44','hyderabad':'NH-44',
    'bangalore':'NH-44','bengaluru':'NH-44','chennai':'NH-44',
    'mumbai':'NH-48','pune':'NH-48','hubli':'NH-48',
    'vijayawada':'NH-16','visakhapatnam':'NH-16','nellore':'NH-16',
    'mysore':'NH-275','coimbatore':'NH-275','kochi':'NH-275',
    'thiruvananthapuram':'NH-275','trivandrum':'NH-275',
    'hyderabad':'NH-65','kurnool':'NH-65','solapur':'NH-65',
}

def get_season_bonus(city, month):
    city_lower = str(city).lower().strip()
    highway = 'NH-44'
    for key,hw in city_highway_map.items():
        if key in city_lower:
            highway = hw
            break
    bonus = 0.0
    if highway == 'NH-44' and month in [6,7,8,9]:   bonus = 0.3
    elif highway == 'NH-16' and month in [10,11]:    bonus = 0.4
    elif highway == 'NH-275' and month in [6,7,8,9]: bonus = 0.2
    return bonus

df['month'] = pd.to_datetime(df['date'], errors='coerce').dt.month.fillna(6).astype(int)

records = []
for _, row in df.iterrows():
    rainfall   = map_rainfall(row['weather'])
    temp       = float(row['temperature'])
    humidity   = visibility_map.get(str(row['visibility']).lower().strip(), 65.0)
    congestion = traffic_map.get(str(row['traffic_density']).lower().strip(), 0.5)
    news_risk  = map_cause(row['cause'])
    month      = int(row['month'])
    city       = str(row['city'])
    bonus      = get_season_bonus(city, month)
    severity   = str(row['accident_severity']).lower().strip()
    disruption = severity_map.get(severity, 0)

    if rainfall > 80:                        disruption = 1
    if congestion > 0.7 and news_risk >= 1:  disruption = 1

    records.append({
        'rainfall_mm':          rainfall,
        'temp_c':               temp,
        'humidity':             humidity,
        'congestion_level':     congestion,
        'news_risk_count':      news_risk,
        'month':                month,
        'highway_season_bonus': bonus,
        'disruption':           disruption,
    })

df_real = pd.DataFrame(records)
print(f"Real records: {len(df_real)}")
print(df_real['disruption'].value_counts())

In [ ]:
np.random.seed(42)
highways = ['NH-44','NH-48','NH-16','NH-275','NH-65']
synthetic = []

for _ in range(4000):
    highway = np.random.choice(highways)
    month   = np.random.randint(1,13)

    if month in [6,7,8,9]:
        rainfall = np.random.choice([
            np.random.uniform(0,20),
            np.random.uniform(20,80),
            np.random.uniform(80,180)
        ], p=[0.15,0.35,0.50])
        humidity = np.random.uniform(70,98)
    elif month in [10,11]:
        rainfall = np.random.choice([
            np.random.uniform(0,10),
            np.random.uniform(10,60),
            np.random.uniform(60,150)
        ], p=[0.25,0.40,0.35])
        humidity = np.random.uniform(60,92)
    else:
        rainfall = np.random.choice([
            np.random.uniform(0,5),
            np.random.uniform(5,30),
        ], p=[0.75,0.25])
        humidity = np.random.uniform(30,65)

    congestion = np.random.uniform(0,1)
    news_risk  = np.random.randint(0,4)
    temp       = np.random.uniform(18,42)
    bonus      = 0.0
    if highway == 'NH-44' and month in [6,7,8,9]:   bonus = 0.3
    elif highway == 'NH-16' and month in [10,11]:    bonus = 0.4
    elif highway == 'NH-275' and month in [6,7,8,9]: bonus = 0.2

    disruption = 0
    if rainfall > 80:                        disruption = 1
    elif rainfall > 40 and humidity > 75:    disruption = 1
    elif news_risk >= 2:                     disruption = 1
    elif congestion > 0.7:                   disruption = 1
    elif bonus > 0 and rainfall > 20:        disruption = 1

    synthetic.append({
        'rainfall_mm':          round(rainfall,2),
        'temp_c':               round(temp,2),
        'humidity':             round(humidity,2),
        'congestion_level':     round(congestion,2),
        'news_risk_count':      news_risk,
        'month':                month,
        'highway_season_bonus': bonus,
        'disruption':           disruption,
    })

df_syn = pd.DataFrame(synthetic)
df_all = pd.concat([df_real, df_syn], ignore_index=True)

print(f"Total: {len(df_all)} | Real: {len(df_real)} | Synthetic: {len(df_syn)}")
print(df_all['disruption'].value_counts())
print(f"Disruption rate: {df_all['disruption'].mean():.2%}")

In [ ]:
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
import mlflow, pickle

# EXACT 7 features — must match spark_job.py
FEATURES = ['rainfall_mm','temp_c','humidity','congestion_level',
            'news_risk_count','month','highway_season_bonus']

X = df_all[FEATURES]
y = df_all['disruption']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Features: {FEATURES}")

mlflow.set_experiment("freight-ensemble-nhai-v4-clean")

with mlflow.start_run():
    # XGBoost
    xgb_model = xgb.XGBClassifier(
        n_estimators=300, max_depth=6,
        learning_rate=0.03, subsample=0.8,
        colsample_bytree=0.8, min_child_weight=3,
        eval_metric='logloss', random_state=42
    )
    xgb_model.fit(X_train, y_train)
    xgb_proba = xgb_model.predict_proba(X_test)[:,1]
    xgb_pred  = (xgb_proba >= 0.5).astype(int)
    xgb_acc   = accuracy_score(y_test, xgb_pred)
    xgb_f1    = f1_score(y_test, xgb_pred)
    xgb_auc   = roc_auc_score(y_test, xgb_proba)

    # LightGBM
    lgb_model = lgb.LGBMClassifier(
        n_estimators=300, max_depth=6,
        learning_rate=0.03, subsample=0.8,
        min_child_samples=20, random_state=42, verbose=-1
    )
    lgb_model.fit(X_train, y_train)
    lgb_proba = lgb_model.predict_proba(X_test)[:,1]
    lgb_pred  = (lgb_proba >= 0.5).astype(int)
    lgb_acc   = accuracy_score(y_test, lgb_pred)
    lgb_f1    = f1_score(y_test, lgb_pred)
    lgb_auc   = roc_auc_score(y_test, lgb_proba)

    # RandomForest
    rf_model = RandomForestClassifier(
        n_estimators=300, max_depth=8,
        min_samples_split=10, min_samples_leaf=5,
        random_state=42, n_jobs=-1
    )
    rf_model.fit(X_train, y_train)
    rf_proba = rf_model.predict_proba(X_test)[:,1]
    rf_pred  = (rf_proba >= 0.5).astype(int)
    rf_acc   = accuracy_score(y_test, rf_pred)
    rf_f1    = f1_score(y_test, rf_pred)
    rf_auc   = roc_auc_score(y_test, rf_proba)

    # Ensemble
    ens_proba = xgb_proba*0.5 + lgb_proba*0.3 + rf_proba*0.2
    ens_pred  = (ens_proba >= 0.5).astype(int)
    ens_acc   = accuracy_score(y_test, ens_pred)
    ens_f1    = f1_score(y_test, ens_pred)
    ens_auc   = roc_auc_score(y_test, ens_proba)

    print("="*60)
    print(f"{'Model':<15} {'Accuracy':>10} {'F1':>10} {'AUC':>10}")
    print("="*60)
    print(f"{'XGBoost':<15} {xgb_acc:>10.4f} {xgb_f1:>10.4f} {xgb_auc:>10.4f}")
    print(f"{'LightGBM':<15} {lgb_acc:>10.4f} {lgb_f1:>10.4f} {lgb_auc:>10.4f}")
    print(f"{'RandomForest':<15} {rf_acc:>10.4f} {rf_f1:>10.4f} {rf_auc:>10.4f}")
    print(f"{'ENSEMBLE':<15} {ens_acc:>10.4f} {ens_f1:>10.4f} {ens_auc:>10.4f}")
    print("="*60)
    print(classification_report(y_test, ens_pred,
          target_names=['No Disruption','Disruption']))

    cv = cross_val_score(xgb_model, X, y, cv=5, scoring='f1')
    print(f"XGB 5-fold CV F1: {cv.mean():.4f} ± {cv.std():.4f}")

    mlflow.log_param("total_records",    len(df_all))
    mlflow.log_param("real_nhai",        len(df_real))
    mlflow.log_param("synthetic",        len(df_syn))
    mlflow.log_param("features",         str(FEATURES))
    mlflow.log_param("ensemble_weights", "0.5/0.3/0.2")
    mlflow.log_metric("xgb_accuracy",    xgb_acc)
    mlflow.log_metric("lgb_accuracy",    lgb_acc)
    mlflow.log_metric("rf_accuracy",     rf_acc)
    mlflow.log_metric("ens_accuracy",    ens_acc)
    mlflow.log_metric("ens_f1",          ens_f1)
    mlflow.log_metric("ens_auc",         ens_auc)
    mlflow.log_metric("cv_f1_mean",      cv.mean())
    mlflow.log_metric("cv_f1_std",       cv.std())

with open('xgb_model_v3.pkl','wb') as f: pickle.dump(xgb_model, f)
with open('lgb_model_v3.pkl','wb') as f: pickle.dump(lgb_model, f)
with open('rf_model_v3.pkl', 'wb') as f: pickle.dump(rf_model,  f)
print("\nAll 3 models saved. Features:", FEATURES)

In [ ]:
from google.colab import files
files.download('xgb_model_v3.pkl')
files.download('lgb_model_v3.pkl')
files.download('rf_model_v3.pkl')